# Multitask RNN (Yang et al., 2019)

This notebook reproduces the core analyses from:

> Yang, G. R., Joglekar, M. R., Song, H. F., Newsome, W. T., & Wang, X. J. (2019).
> Task representations in neural networks trained to perform many cognitive tasks.
> *Nature Neuroscience*, 22(2), 297-306.

The paper trains a single recurrent neural network to perform **20 inter-related cognitive tasks** and studies how task representations are organized in the network. We will:

1. Build the 20-task dataset with NeuralRNN's data layer.
2. Train a CTRNN on all tasks simultaneously.
3. Reproduce Figures 2, 4, 5, and 6 from the paper.

In [ ]:
import os
import warnings
import sys
sys.path.append('../src')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import torch

from neuralrnn.models.ctrnn.configuration_ctrnn import CTRNNConfig
from neuralrnn.models.ctrnn.modeling_ctrnn import CTRNNModel
from neuralrnn.train.trainer import Trainer
from neuralrnn.train.training_args import TrainingArguments
from neuralrnn.train.objectives.supervised import SupervisedObjective
from neuralrnn.data.tasks.multitask_yang_task import generate_trials, RULES_ALL, RULE_NAME
from neuralrnn.data.tasks.multitask_yang_dataset import MultitaskYangDataset

%matplotlib inline
warnings.filterwarnings("ignore")

DT = 20.0
TAU = 100.0
INPUT_DIM = 85
OUTPUT_DIM = 33
N_EACHRING = 32
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## 1. The 20-task dataset

The dataset uses a shared input/output architecture:

- Fixation input (1 dim): 1 during fixation, 0 during response.
- Stimulus rings (2 x 32 dims): Gaussian population code for circular variables.
- Rule input (20 dims): one-hot task indicator.
- Fixation output (1 dim): target 0.85 during fixation, 0.05 during response.
- Output ring (32 dims): Gaussian population code for response direction.

The 20 tasks fall into five families: Go, Anti, DM, Delayed DM, and Matching.

In [ ]:
def plot_example_trial(rule, seed=0):
    inputs, targets, mask, conds = generate_trials(rule, n_trials=1, mode="random", seed=seed)
    print(inputs.shape)
    inputs = inputs[0]
    targets = targets[0]
    t = np.arange(inputs.shape[0]) * DT

    fig, axes = plt.subplots(5, 1, figsize=(6, 6))
    axes[0].plot(t, inputs[:, 0], color="black", label="fixation input")
    axes[0].set_ylabel("Fixation input")
    axes[0].set_ylim(-0.1, 1.2)
    # axes[1].plot(t, inputs[:, 1:1 + N_EACHRING].sum(axis=1), color="C0")
    axes[1].imshow(inputs[:, 1:1 + N_EACHRING].T, aspect="auto", cmap="viridis")
    axes[1].set_ylabel("Modality 1 ring")
    # axes[2].plot(t, inputs[:, 1 + N_EACHRING:1 + 2 * N_EACHRING].sum(axis=1), color="C1")
    axes[2].imshow(inputs[:, 1 + N_EACHRING:1 + 2 * N_EACHRING].T, aspect="auto", cmap="viridis")
    axes[2].set_ylabel("Modality 2 ring")
    rule_idx = int(np.argmax(inputs[0, 1 + 2 * N_EACHRING:]))
    axes[3].axhline(rule_idx, color="C2", linewidth=2)
    axes[3].set_ylabel("Rule index")
    axes[3].set_ylim(-0.5, 20.5)
    axes[4].plot(t, targets[:, 0], color="black", label="fixation target")
    # axes[4].plot(t, targets[:, 1:].sum(axis=1), color="C3", label="output ring sum")
    axes[4].imshow(targets[:, 1:].T, aspect="auto", cmap="viridis")
    axes[4].set_ylabel("Target output")
    axes[4].set_xlabel("Time (ms)")
    
    fig.suptitle(f"Example trial: {RULE_NAME.get(rule, rule)} ({rule})")
    plt.tight_layout()
    plt.show()

for r in ["fdgo", "dm1", "contextdm1", "delaygo", "dmsgo"]:
    plot_example_trial(r, seed=0)

## 2. Train a CTRNN on all 20 tasks

We use a leaky CTRNN with Softplus activation, matching the paper. The recurrent weights are initialized to 0.5 * I. Tasks are interleaved; contextdm1 and contextdm2 are oversampled 5x.

In [ ]:
MODEL_DIR = "models/12/yang_multitask"
os.makedirs(MODEL_DIR, exist_ok=True)

config = CTRNNConfig(
    input_dim=INPUT_DIM,
    latent_dim=256,
    output_dim=OUTPUT_DIM,
    dt=DT,
    tau=TAU,
    activation="softplus",
    sigma_rec=0.02,
    noise_alpha_scaling=True,
    trainable_h0=False,
)

model = CTRNNModel(config).to(DEVICE)

with torch.no_grad():
    torch.nn.init.eye_(model.h2h.weight)
    model.h2h.weight *= 0.5
    model.input2h.weight.normal_(0, 1.0 / np.sqrt(INPUT_DIM))
    model.readout_layer.weight.normal_(0, 0.4 / np.sqrt(config.latent_dim))
    model.readout_layer.bias.zero_()

print("Parameters:", sum(p.numel() for p in model.parameters()))

In [ ]:
train_ds = MultitaskYangDataset(
    rules=RULES_ALL,
    rule_prob_map={"contextdm1": 10.0, "contextdm2": 10.0},
    batch_size=512,
    mode="random",
    sigma_x=0.01,
    seed=0,
)

objective = SupervisedObjective(task_type="regression")
args = TrainingArguments(
    output_dir=MODEL_DIR,
    max_steps=20000,
    batch_size=512,
    learning_rate=0.001,
    log_every=500,
    save_every=500,
    device=str(DEVICE),
    seed=0,
)

trainer = Trainer(model=model, dataset=train_ds, objective=objective, args=args)

In [ ]:
import glob
overwrite = True
ckpt_files = glob.glob(os.path.join(MODEL_DIR, "model.safetensors"))
if ckpt_files and os.path.exists(ckpt_files[0]) and not overwrite:
    print("Loading existing checkpoint from", MODEL_DIR)
    model = CTRNNModel.from_pretrained(MODEL_DIR)
    model.to(DEVICE)
else:
    print("Training from scratch...")
    history = trainer.train()
    model.save_pretrained(MODEL_DIR)
    print("Saved to", MODEL_DIR)

## 3. Evaluation

A trial is correct if the network maintains fixation when required, or responds within 0.2*pi of the target direction.

In [ ]:
def popvec(y):
    '''Population-vector decode a circular direction from an output ring.'''
    pref = np.arange(0, 2 * np.pi, 2 * np.pi / y.shape[-1])
    if y.ndim == 3:
        y = y[-1]
    temp_sum = y.sum(axis=-1)
    temp_cos = np.sum(y * np.cos(pref), axis=-1) / temp_sum
    temp_sin = np.sum(y * np.sin(pref), axis=-1) / temp_sum
    loc = np.arctan2(temp_sin, temp_cos)
    return np.mod(loc, 2 * np.pi)


def get_perf(y_hat, y_loc):
    '''Compute trial-by-trial performance.

    Args:
        y_hat: (B, T, output_dim) network output.
        y_loc: (B,) target response location, -1 for fixation (no-go) trials.
    '''
    y_hat = y_hat[:, -1, :]  # last time step
    y_hat_fix = y_hat[..., 0]
    y_hat_loc = popvec(y_hat[..., 1:])
    fixating = y_hat_fix > 0.5
    original_dist = y_loc - y_hat_loc
    dist = np.minimum(np.abs(original_dist), 2 * np.pi - np.abs(original_dist))
    corr_loc = dist < 0.2 * np.pi
    should_fix = y_loc < 0
    perf = should_fix * fixating + (1 - should_fix) * corr_loc * (1 - fixating)
    return perf


def evaluate_task(model, rule, n_trials=512):
    model.eval()
    inputs, targets, mask, conds = generate_trials(rule, n_trials=n_trials, mode="test", seed=0)
    inputs_t = torch.from_numpy(inputs).to(DEVICE)
    with torch.no_grad():
        y_hat = model(inputs_t).outputs.cpu().numpy()
    # The true response location is stored in condition metadata.
    y_loc = np.array([c["response_loc"] for c in conds])
    perf = get_perf(y_hat, y_loc)
    return float(perf.mean())


perf_by_task = {rule: evaluate_task(model, rule) for rule in RULES_ALL}

fig, ax = plt.subplots(figsize=(12, 4))
rules_short = [RULE_NAME.get(r, r) for r in RULES_ALL]
vals = [perf_by_task[r] for r in RULES_ALL]
ax.bar(range(len(RULES_ALL)), vals, color="C0")
ax.set_xticks(range(len(RULES_ALL)))
ax.set_xticklabels(rules_short, rotation=45, ha="right")
ax.set_ylabel("Performance")
ax.set_ylim(0, 1.05)
ax.axhline(0.95, color="red", linestyle="--", label="target")
ax.set_title("Performance on all 20 tasks")
ax.legend()
plt.tight_layout()
plt.show()

print("Mean:", np.mean(list(perf_by_task.values())))
print("Min:", np.min(list(perf_by_task.values())))

In [ ]:
# plot the accuracy of each checkpoint
fig, ax = plt.subplots(figsize=(6, 3))
vals = []
for i in range(500, 20000, 500):
    _DIR = os.path.join(MODEL_DIR, f"checkpoint-{i}")
    _model = CTRNNModel.from_pretrained(_DIR)
    _model.to(DEVICE)
    perf_by_task = {rule: evaluate_task(_model, rule, n_trials=128) for rule in RULES_ALL}
    vals.append([perf_by_task[r] for r in RULES_ALL])
    print(f"Checkpoint {i}: mean={np.mean(list(perf_by_task.values()))}")
vals = np.array(vals)
for i, r in enumerate(RULES_ALL):
    ax.plot(range(500, 20000, 500), vals[:, i], label=RULE_NAME.get(r, r))
plt.show()
    

## 4. Reproduce Figure 2 — Functional clusters

Task variance of a unit for a task is the variance of its activity across stimulus conditions, averaged over time (excluding fixation).

In [ ]:
def compute_task_variance(model, rules=RULES_ALL, n_conditions=16, active_thresh=1e-3):
    tv = {}
    for rule in rules:
        model.eval()
        inputs, targets, mask, conds = generate_trials(rule, n_trials=n_conditions, mode="test", seed=0)
        inputs_t = torch.from_numpy(inputs).to(DEVICE)
        with torch.no_grad():
            states = model(inputs_t).states.cpu().numpy()
        stim_on = next((v[1] for k, v in conds[0]["epochs"].items() if k.startswith("stim")), 5)
        states_resp = states[:, stim_on:, :]
        tv[rule] = states_resp.var(axis=0).mean(axis=0)
    tv_mat = np.stack([tv[r] for r in rules], axis=1)
    tv_norm = tv_mat / (tv_mat.max(axis=1, keepdims=True) + 1e-10)
    active = tv_mat.sum(axis=1) > active_thresh
    return tv, tv_mat, tv_norm, active


tv, tv_mat, tv_norm, active = compute_task_variance(model, n_conditions=16)
print("Active units:", active.sum(), "/", len(active))

In [ ]:
def cluster_units(tv_norm, active, k_range=range(2, 31)):
    X = tv_norm[active]
    best_score, best_k, best_labels = -1, 2, None
    for k in k_range:
        labels = KMeans(n_clusters=k, random_state=0, n_init=10).fit_predict(X)
        score = silhouette_score(X, labels)
        if score > best_score:
            best_score, best_k, best_labels = score, k, labels
    print(f"Optimal clusters: {best_k}, silhouette={best_score:.3f}")
    return best_labels, best_k

labels, n_clusters = cluster_units(tv_norm, active)
order = np.argsort(labels)
sorted_tv = tv_norm[active][order]

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(sorted_tv, aspect="auto", cmap="viridis", vmin=0, vmax=1)
ax.set_yticks([])
ax.set_xticks(range(len(RULES_ALL)))
ax.set_xticklabels([RULE_NAME.get(r, r) for r in RULES_ALL], rotation=45, ha="right")
ax.set_ylabel("Units (sorted by cluster)")
ax.set_title("Normalized task variance (Fig. 2c-like)")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
X = tv_norm[active]
embed = TSNE(n_components=2, random_state=0, perplexity=30).fit_transform(X)

fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(embed[:, 0], embed[:, 1], c=labels, cmap="tab20", s=20)
ax.set_title("t-SNE of task variance (Fig. 2d-like)")
plt.colorbar(scatter, ax=ax, label="Cluster")
plt.tight_layout()
plt.show()

In [ ]:
def lesion_cluster(model, cluster_unit_indices):
    lesioned = CTRNNModel(model.config).to(DEVICE)
    lesioned.load_state_dict(model.state_dict())
    with torch.no_grad():
        lesioned.h2h.weight[:, cluster_unit_indices] = 0.0
        lesioned.readout_layer.weight[:, cluster_unit_indices] = 0.0
    return lesioned


cluster_perf_change = []
full_perf = np.array([perf_by_task[r] for r in RULES_ALL])
for c in range(n_clusters):
    unit_idx = np.where(labels == c)[0]
    global_idx = np.where(active)[0][unit_idx]
    lesioned = lesion_cluster(model, global_idx)
    lesioned_perf = np.array([evaluate_task(lesioned, r, n_trials=256) for r in RULES_ALL])
    cluster_perf_change.append(lesioned_perf - full_perf)
cluster_perf_change = np.array(cluster_perf_change)

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(cluster_perf_change, aspect="auto", cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(RULES_ALL)))
ax.set_xticklabels([RULE_NAME.get(r, r) for r in RULES_ALL], rotation=45, ha="right")
ax.set_yticks(range(n_clusters))
ax.set_ylabel("Lesioned cluster")
ax.set_title("Performance change after cluster lesion (Fig. 2e-like)")
plt.colorbar(im, ax=ax, label="Delta Performance")
plt.tight_layout()
plt.show()

## 5. Reproduce Figure 4 — Pairwise FTV distributions

Fractional task variance (FTV) measures whether a unit is selective for one task over another.

In [ ]:
def compute_ftv(tv, task_a, task_b):
    tva = tv[task_a][active]
    tvb = tv[task_b][active]
    return (tva - tvb) / (tva + tvb + 1e-10)


pairs = [
    ("fdanti", "dm1", "disjoint"),
    ("delaydm1", "dm1", "inclusive"),
    ("delaygo", "dm1", "mixed"),
    ("contextdm1", "contextdm2", "disjoint-equal"),
    ("dm1", "multidm", "disjoint-mixed"),
]

fig, axes = plt.subplots(1, len(pairs), figsize=(16, 3))
for ax, (a, b, label) in zip(axes, pairs):
    ftv = compute_ftv(tv, a, b)
    ax.hist(ftv, bins=30, color="C0", edgecolor="white")
    ax.set_title(f"{RULE_NAME[a]} vs {RULE_NAME[b]}({label})")
    ax.set_xlim(-1, 1)
    ax.set_xlabel("FTV")
    ax.set_ylabel("# units")
plt.suptitle("Fractional task variance (Fig. 4a-e-like)")
plt.tight_layout()
plt.show()

## 6. Reproduce Figure 5 — Context-dependent DM dissection

Classify units by FTV(contextdm1, contextdm2) into three groups and measure lesion effects.

In [ ]:
ftv_ctx = compute_ftv(tv, "contextdm1", "contextdm2")
threshold = 0.8
group1 = np.where(ftv_ctx > threshold)[0]
group2 = np.where(ftv_ctx < -threshold)[0]
group12 = np.where((ftv_ctx > -0.1) & (ftv_ctx < 0.1))[0]

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(ftv_ctx, bins=40, color="lightgray", edgecolor="black")
ax.axvline(threshold, color="C0", linestyle="--", label="group 1")
ax.axvline(-threshold, color="C1", linestyle="--", label="group 2")
ax.axvspan(-0.1, 0.1, alpha=0.3, color="C2", label="group 12")
ax.set_xlabel("FTV(Ctx DM 1, Ctx DM 2)")
ax.set_ylabel("# units")
ax.set_title("Context DM grouping (Fig. 5a-like)")
ax.legend()
plt.tight_layout()
plt.show()
print(f"Group1: {len(group1)}, Group2: {len(group2)}, Group12: {len(group12)}")

In [ ]:
full_perf_ctx = np.array([perf_by_task[r] for r in ["contextdm1", "contextdm2", "dm1", "dm2", "multidm"]])
groups = {"group1": group1, "group2": group2, "group12": group12}
group_lesion_change = {}
for name, g in groups.items():
    global_idx = np.where(active)[0][g]
    lesioned = lesion_cluster(model, global_idx)
    lesioned_perf = np.array([evaluate_task(lesioned, r, n_trials=256) for r in ["contextdm1", "contextdm2", "dm1", "dm2", "multidm"]])
    group_lesion_change[name] = lesioned_perf # - full_perf_ctx

x = np.arange(5)
width = 0.25
fig, ax = plt.subplots(figsize=(10, 4))
for i, (name, change) in enumerate(group_lesion_change.items()):
    ax.bar(x + i * width, change, width, label=name)
ax.set_xticks(x + width)
ax.set_xticklabels([RULE_NAME[r] for r in ["contextdm1", "contextdm2", "dm1", "dm2", "multidm"]])
ax.set_ylabel("Performance")
ax.set_title("Lesion effects (Fig. 5b-like)")
ax.legend()
plt.tight_layout()
plt.show()

## 7. Reproduce Figure 6 — Compositional task representations

Task vectors are average hidden states at the end of the stimulus epoch. Compositional structure predicts similar difference vectors for shared cognitive components.

In [ ]:
def compute_task_vector(model, rule, n_conditions=32):
    '''Average hidden state at the end of the stimulus epoch.

    Following the original paper, states are first averaged across stimulus
    conditions, then the last time step of the stimulus epoch is extracted.
    Reaction-time tasks (reactgo/reactanti) do not have a separate stim1 epoch,
    so their task vector is taken from the last step of the go epoch.
    '''
    model.eval()
    inputs, targets, mask, conds = generate_trials(rule, n_trials=n_conditions, mode="test", seed=0)
    inputs_t = torch.from_numpy(inputs).to(DEVICE)
    with torch.no_grad():
        states = model(inputs_t).states.cpu().numpy()
    # Average across stimulus conditions.
    states_avg = states.mean(axis=0)
    epochs = conds[0]["epochs"]
    if "stim1" in epochs:
        t_end = epochs["stim1"][1]
    elif "go1" in epochs:
        # Reaction-time tasks: stimulus and response coincide.
        t_end = epochs["go1"][1]
        if t_end is None:
            t_end = states_avg.shape[0]
    else:
        # Fallback: use the last time step before response (if any).
        t_end = states_avg.shape[0]
    return states_avg[t_end - 1, :]


task_vectors = {r: compute_task_vector(model, r) for r in RULES_ALL}
vectors = np.stack([task_vectors[r] for r in RULES_ALL])
pca = PCA(n_components=5)
pcs = pca.fit_transform(vectors)

fig, ax = plt.subplots(figsize=(10, 8))
for i, r in enumerate(RULES_ALL):
    ax.scatter(pcs[i, 0], pcs[i, 1], s=100)
    ax.annotate(RULE_NAME.get(r, r), (pcs[i, 0], pcs[i, 1]), fontsize=8)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
ax.set_title("Task vectors in PC space (Fig. 6-like)")
plt.tight_layout()
plt.show()

In [ ]:
v_go = task_vectors["fdgo"]
v_dlygo = task_vectors["delaygo"]
v_anti = task_vectors["fdanti"]
v_dlyanti = task_vectors["delayanti"]
wm_go = v_dlygo - v_go
wm_anti = v_dlyanti - v_anti
cos1 = np.dot(wm_go, wm_anti) / (np.linalg.norm(wm_go) * np.linalg.norm(wm_anti))

v_ctx1 = task_vectors["contextdm1"]
v_ctx2 = task_vectors["contextdm2"]
v_ctxdly1 = task_vectors["contextdelaydm1"]
v_ctxdly2 = task_vectors["contextdelaydm2"]
ctx_diff = v_ctx2 - v_ctx1
ctxdly_diff = v_ctxdly2 - v_ctxdly1
cos2 = np.dot(ctx_diff, ctxdly_diff) / (np.linalg.norm(ctx_diff) * np.linalg.norm(ctxdly_diff))

print(f"WM component cosine similarity: {cos1:.3f}")
print(f"Context component cosine similarity: {cos2:.3f}")

## 8. Summary

We built the Yang et al. (2019) 20-task dataset, trained a CTRNN, and reproduced the main representational analyses:

- **Fig. 2**: Task-variance clustering.
- **Fig. 4**: Pairwise FTV distributions.
- **Fig. 5**: Context-dependent DM dissection.
- **Fig. 6**: Compositional task vectors.

For paper-level performance, train for more steps (the original used up to 1e7 trials). The analysis pipeline is identical once a trained checkpoint is available.